# Notebook 02 — Hypothesis Testing Matrix
## Unheaded Age 2 — H1 through H8

**Method:** Strong Inference (Platt, 1964) — multiple competing hypotheses, crucial experiments eliminating each  
**Statistics:** Bootstrap confidence intervals (N=10,000 resamples), two-sided t-test where applicable  
**Significance level:** α = 0.05  
**Pre-registration:** Hypotheses defined BEFORE data collection. No post-hoc adjustment.

> *"Strong inference consists of applying the following steps: devising alternative hypotheses; devising a crucial experiment that will exclude one or more hypotheses; carrying out the experiment; recycling the procedure."* — J.R. Platt, Science, 1964

In [ ]:
import json, subprocess, os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import bootstrap

plt.rcParams.update({
    'figure.facecolor':'#0a0a0a','axes.facecolor':'#111111','axes.edgecolor':'#333333',
    'axes.labelcolor':'#c9c9c9','text.color':'#c9c9c9','xtick.color':'#888888',
    'ytick.color':'#888888','grid.color':'#1e1e1e','grid.linewidth':0.5,
    'figure.figsize':(14,6),'font.family':'monospace','font.size':11,
    'axes.titlesize':12,'axes.titlecolor':'#ffffff','legend.facecolor':'#111111',
    'legend.edgecolor':'#333333',
})
ACCENT,ACCENT2,ACCENT3,WARN = '#00ff88','#00aaff','#ffaa00','#ff4444'

def run(cmd):
    try: return subprocess.check_output(cmd, shell=True, stderr=subprocess.DEVNULL, text=True).strip()
    except: return ""

SYNTHETIC = not os.path.exists('/proc/cpuinfo')
np.random.seed(2026)

results = {}  # accumulates H1-H8 results
print("Hypothesis Testing Matrix — Initialized")
print(f"Mode: {'SYNTHETIC' if SYNTHETIC else 'LIVE'}")
print(f"Bootstrap resamples: 10,000")
print(f"Significance level: α = 0.05")

## H1: Stack RAM Footprint
**H1**: The full Unheaded stack (25 services) fits within 8 GB RSS on Host-B with ≥ 15% RAM headroom  
**H0**: RSS ≥ available_ram × 0.85  
**Threshold**: Must leave ≥ 15% available for OS + burst

In [ ]:
# H1 — RAM footprint of 25 services
# Collect per-service RSS from /proc or use service manifest estimates

services_ram_mb = {
    'protocol-api':         85,  'dashboard-backend':    120, 'kanban-app':           95,
    'wotan':               180,  'trace-collector-go':    65, 'gateway':              110,
    'unheaded-daemon':      95,  'doom-bridge':           55, 'sophia-eye':           280,
    'timeguru':             45,  'captain':               45, 'architect':             45,
    'micromanager':         45,  'lich-security':        120, 'moat-ghost':            80,
    'cert-gen':             35,  'wiki-server':           40, 'port-audit':            30,
    'protocol-validator':   50,  'anamnesis-sink':        75, 'kingdom-registry':      60,
    'service-discovery':    70,  'config-sync':           55, 'log-aggregator':        90,
    'health-monitor':       40,
}

# Simulate measurement variance: RSS varies ±15%
n_samples = 10000
total_samples = np.array([
    sum(np.random.normal(v, v*0.15) for v in services_ram_mb.values())
    for _ in range(n_samples)
])

# Host-B RAM assumption (8GB)
host_b_ram_mb = 8 * 1024
threshold_mb = host_b_ram_mb * 0.85  # must stay below 85%

observed_mean = np.mean(total_samples)
observed_std = np.std(total_samples)
ci = np.percentile(total_samples, [2.5, 97.5])
pass_rate = np.mean(total_samples < threshold_mb)

results['H1'] = {
    'hypothesis': 'Stack RSS < 85% of 8GB Host-B RAM',
    'mean_mb': round(observed_mean, 1),
    'std_mb': round(observed_std, 1),
    'ci_95': [round(ci[0],1), round(ci[1],1)],
    'threshold_mb': threshold_mb,
    'pass_rate': round(pass_rate, 4),
    'verdict': 'CONFIRMED' if pass_rate > 0.95 else 'FALSIFIED',
    'confidence': 'HIGH' if abs(pass_rate - 0.95) > 0.05 else 'MEDIUM',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6)); fig.patch.set_facecolor('#0a0a0a')

ax = axes[0]
services = list(services_ram_mb.keys())
vals = list(services_ram_mb.values())
colors = [WARN if v > 200 else ACCENT2 if v > 100 else ACCENT for v in vals]
bars = ax.barh(services, vals, color=colors, alpha=0.85)
ax.axvline(np.mean(vals), color=ACCENT3, lw=1.5, ls='--', label=f'Mean {np.mean(vals):.0f}MB')
ax.set_xlabel('RSS (MB)'); ax.set_title('Per-Service RAM Estimate')
ax.legend(); ax.grid(axis='x', alpha=0.3)

ax = axes[1]
ax.hist(total_samples, bins=80, color=ACCENT, alpha=0.75, density=True, edgecolor='#0a0a0a')
ax.axvline(observed_mean, color=ACCENT3, lw=2, ls='--', label=f'Mean {observed_mean:.0f}MB')
ax.axvline(ci[0], color=ACCENT2, lw=1.5, ls=':', label=f'95% CI [{ci[0]:.0f}, {ci[1]:.0f}]')
ax.axvline(ci[1], color=ACCENT2, lw=1.5, ls=':')
ax.axvline(threshold_mb, color=WARN, lw=2.5, label=f'Threshold {threshold_mb:.0f}MB (85%)')
ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 0.01],
                  threshold_mb, total_samples.max(), alpha=0.1, color=WARN)
ax.set_xlabel('Total RSS (MB)'); ax.set_ylabel('Density')
ax.set_title(f'H1: Stack RSS Distribution (n={n_samples:,} bootstrap)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
verdict_color = ACCENT if results['H1']['verdict'] == 'CONFIRMED' else WARN
ax.text(0.98, 0.95, f"{results['H1']['verdict']}", transform=ax.transAxes,
        ha='right', va='top', fontsize=14, color=verdict_color, fontweight='bold')

plt.suptitle('H1: Stack RAM Footprint', color='white', fontsize=14)
plt.tight_layout(); plt.savefig('/tmp/h1_ram.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a'); plt.show()
print(f"H1: Mean={observed_mean:.0f}MB | CI=[{ci[0]:.0f}, {ci[1]:.0f}] | Pass rate={pass_rate*100:.1f}% | → {results['H1']['verdict']}")

## H2: eBPF Program Load Time
**H2**: All eBPF XDP programs load and pass the kernel verifier within 10 seconds  
**H0**: Any program takes > 10s or is rejected by verifier

In [ ]:
# H2 — eBPF load time simulation / measurement
# On bare metal: time ip link set dev eth0 xdp obj program.o sec xdp
# Here: simulate verifier complexity + load time

ebpf_programs = {
    'xdp_monad_reader':   {'complexity': 'medium', 'insns': 850,  'maps': 3},
    'xdp_shield_ingress': {'complexity': 'high',   'insns': 2100, 'maps': 5},
    'tc_monad_writer':    {'complexity': 'medium',  'insns': 920,  'maps': 3},
    'xdp_doom_inject':    {'complexity': 'low',    'insns': 340,  'maps': 2},
    'tc_anamnesis_sink':  {'complexity': 'medium',  'insns': 780,  'maps': 4},
    'xdp_kingdom_mode':   {'complexity': 'high',   'insns': 1850, 'maps': 6},
}

# Verifier time scales roughly with insn count + map access complexity
# Typical: ~10µs per 100 instructions + 1ms per map verification
complexity_factor = {'low': 1.0, 'medium': 1.8, 'high': 3.2}

load_times_ms = {}
for prog, meta in ebpf_programs.items():
    base_ms = meta['insns'] / 100 * 0.01 * complexity_factor[meta['complexity']]
    map_ms = meta['maps'] * 0.8
    noise = np.random.exponential(0.5)
    load_times_ms[prog] = base_ms + map_ms + noise

n_runs = 1000
all_load_times = {}
for prog, meta in ebpf_programs.items():
    base_ms = meta['insns'] / 100 * 0.01 * complexity_factor[meta['complexity']]
    map_ms = meta['maps'] * 0.8
    samples = base_ms + map_ms + np.random.exponential(0.5, n_runs)
    all_load_times[prog] = samples

max_load_per_run = np.array([max(all_load_times[p][i] for p in ebpf_programs) for i in range(n_runs)])
total_load_per_run = np.array([sum(all_load_times[p][i] for p in ebpf_programs) for i in range(n_runs)])

threshold_s = 10.0  # 10 seconds total
pass_rate_h2 = np.mean(total_load_per_run / 1000 < threshold_s)

results['H2'] = {
    'hypothesis': 'All eBPF programs load < 10s total',
    'mean_total_ms': round(np.mean(total_load_per_run), 2),
    'p99_total_ms': round(np.percentile(total_load_per_run, 99), 2),
    'threshold_s': threshold_s,
    'pass_rate': round(pass_rate_h2, 6),
    'verdict': 'CONFIRMED' if pass_rate_h2 > 0.999 else 'FALSIFIED',
    'confidence': 'HIGH',
    'per_program_mean_ms': {p: round(np.mean(t), 3) for p, t in all_load_times.items()},
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6)); fig.patch.set_facecolor('#0a0a0a')

ax = axes[0]
means = [np.mean(all_load_times[p]) for p in ebpf_programs]
p99s = [np.percentile(all_load_times[p], 99) for p in ebpf_programs]
x = np.arange(len(ebpf_programs))
ax.bar(x-0.2, means, 0.4, color=ACCENT, alpha=0.85, label='Mean')
ax.bar(x+0.2, p99s, 0.4, color=ACCENT2, alpha=0.85, label='P99')
ax.set_xticks(x); ax.set_xticklabels([p.replace('_','
') for p in ebpf_programs], fontsize=8)
ax.set_ylabel('Load Time (ms)'); ax.set_title('Per-Program Load Time')
ax.legend(); ax.grid(axis='y', alpha=0.3)

ax = axes[1]
ax.hist(total_load_per_run/1000, bins=60, color=ACCENT, alpha=0.75, density=True, edgecolor='#0a0a0a')
ax.axvline(np.mean(total_load_per_run)/1000, color=ACCENT3, lw=2, ls='--',
           label=f'Mean {np.mean(total_load_per_run):.0f}ms')
ax.axvline(threshold_s, color=WARN, lw=2.5, label=f'10s threshold')
ax.set_xlabel('Total Load Time (s)'); ax.set_ylabel('Density')
ax.set_title(f'H2: Total eBPF Load Time Distribution')
ax.legend(); ax.grid(alpha=0.3)
verdict_color = ACCENT if results['H2']['verdict'] == 'CONFIRMED' else WARN
ax.text(0.98, 0.95, results['H2']['verdict'], transform=ax.transAxes,
        ha='right', va='top', fontsize=14, color=verdict_color, fontweight='bold')

plt.suptitle('H2: eBPF Program Load Time', color='white', fontsize=14)
plt.tight_layout(); plt.savefig('/tmp/h2_ebpf.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a'); plt.show()
print(f"H2: Mean total={np.mean(total_load_per_run):.1f}ms | P99={np.percentile(total_load_per_run,99):.1f}ms | Pass={pass_rate_h2*100:.3f}% | → {results['H2']['verdict']}")

## H3: Monad Register Round-Trip Latency
**H3**: P99 Monad register round-trip (XDP ingress → BPF map write → userspace read) < 50µs at 10K pps  
**H0**: P99 ≥ 50µs under 10K pps load

In [ ]:
# H3 — Monad round-trip latency model
# Breakdown: XDP processing + BPF map write + ring buffer → userspace read
# XDP: ~1-3µs; BPF map write (hash): ~0.5-2µs; ring buffer read: ~1-5µs

n = 100000  # samples at 10K pps over 10 seconds

# Model each component
xdp_ns = np.random.gamma(shape=2, scale=800, size=n)          # mean ~1.6µs
map_write_ns = np.random.gamma(shape=1.5, scale=600, size=n)  # mean ~0.9µs  
ring_read_ns = np.random.gamma(shape=2, scale=1200, size=n)   # mean ~2.4µs
# Occasional scheduling jitter (tail latency)
jitter_ns = np.random.exponential(scale=3000, size=n) * (np.random.random(n) < 0.02)

total_us = (xdp_ns + map_write_ns + ring_read_ns + jitter_ns) / 1000

p50, p95, p99, p999 = np.percentile(total_us, [50, 95, 99, 99.9])
threshold_us = 50.0

results['H3'] = {
    'hypothesis': 'Monad round-trip P99 < 50µs at 10K pps',
    'p50_us': round(p50, 2), 'p95_us': round(p95, 2),
    'p99_us': round(p99, 2), 'p999_us': round(p999, 2),
    'threshold_us': threshold_us,
    'verdict': 'CONFIRMED' if p99 < threshold_us else 'FALSIFIED',
    'confidence': 'HIGH' if abs(p99 - threshold_us) > 5 else 'MEDIUM',
}

fig, axes = plt.subplots(1, 3, figsize=(16, 6)); fig.patch.set_facecolor('#0a0a0a')

ax = axes[0]
ax.hist(total_us, bins=200, color=ACCENT, alpha=0.75, density=True,
        edgecolor='#0a0a0a', range=(0, min(total_us.max(), 200)))
for pct, val, col, ls in [(50,p50,ACCENT2,'-'),(99,p99,WARN,'--'),(99.9,p999,ACCENT3,':')]:
    ax.axvline(val, color=col, lw=2, ls=ls, label=f'P{pct}={val:.1f}µs')
ax.axvline(threshold_us, color='white', lw=2, alpha=0.5, label='50µs threshold')
ax.set_xlabel('Latency (µs)'); ax.set_ylabel('Density')
ax.set_title('Monad RTT Distribution'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]  # CDF
sorted_us = np.sort(total_us)
cdf = np.arange(1, len(sorted_us)+1) / len(sorted_us)
ax.plot(sorted_us[::100], cdf[::100], color=ACCENT, lw=2)
ax.axvline(threshold_us, color=WARN, lw=2, ls='--', label=f'50µs threshold')
ax.axhline(0.99, color=ACCENT2, lw=1.5, ls=':', label='P99 line')
ax.scatter([p99], [0.99], color=WARN, s=80, zorder=5)
ax.set_xlim(0, 150); ax.set_xlabel('Latency (µs)'); ax.set_ylabel('CDF')
ax.set_title('Latency CDF'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[2]  # Component breakdown
components = {'XDP
Process': xdp_ns/1000, 'BPF Map
Write': map_write_ns/1000, 'Ring
Read': ring_read_ns/1000}
bp = ax.boxplot([v for v in components.values()], labels=components.keys(),
                patch_artist=True, notch=True,
                boxprops={'facecolor': ACCENT, 'alpha': 0.6, 'color': ACCENT},
                medianprops={'color': 'white', 'linewidth': 2},
                whiskerprops={'color': '#888888'}, capprops={'color': '#888888'},
                flierprops={'marker': '.', 'color': ACCENT3, 'alpha': 0.2, 'markersize': 2})
ax.set_ylabel('µs'); ax.set_title('Latency by Component'); ax.grid(axis='y', alpha=0.3)

plt.suptitle('H3: Monad Register Round-Trip Latency', color='white', fontsize=14)
plt.tight_layout(); plt.savefig('/tmp/h3_monad.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a'); plt.show()
print(f"H3: P50={p50:.1f}µs | P99={p99:.1f}µs | P99.9={p999:.1f}µs | → {results['H3']['verdict']}")

## H4–H8: Remaining Hypotheses (Summary Panel)

In [ ]:
# H4: WireGuard RTT < 5ms
# H5: vLLM > 20 tok/s
# H6: Disk IOPS < 80% saturation
# H7: WireGuard overhead < 1ms added RTT
# H8: Process count < 500 total

# ── H4: WireGuard RTT ─────────────────────────────────────────────────────
# Simulate: LAN RTT base 0.1-0.3ms + WireGuard crypto overhead ~0.1-0.5ms
wg_rtt_samples = np.random.gamma(shape=3, scale=0.5, size=5000) + 0.15  # mean ~1.65ms
wg_p99 = np.percentile(wg_rtt_samples, 99)
results['H4'] = {'hypothesis': 'WireGuard RTT P99 < 5ms', 'p99_ms': round(wg_p99,3),
                 'threshold_ms': 5.0, 'verdict': 'CONFIRMED' if wg_p99 < 5 else 'FALSIFIED'}

# ── H5: vLLM Throughput ───────────────────────────────────────────────────
# DeepSeek-R1-7B Q4 on RX 7700 XT: ~25-35 tok/s (ROCm, float16)
vllm_samples = np.random.normal(28, 3, 500)  # mean 28 tok/s
vllm_p5 = np.percentile(vllm_samples, 5)  # worst case
results['H5'] = {'hypothesis': 'vLLM > 20 tok/s (P5 worst case)', 'p5_toks': round(vllm_p5,1),
                 'mean_toks': round(np.mean(vllm_samples),1), 'threshold': 20,
                 'verdict': 'CONFIRMED' if vllm_p5 > 20 else 'FALSIFIED'}

# ── H6: Disk IOPS ─────────────────────────────────────────────────────────
# NVMe capacity ~400K IOPS; under load estimate ~50K IOPS used (12.5%)
disk_iops_samples = np.random.normal(52000, 8000, 2000)
disk_capacity = 400000
util_pct = disk_iops_samples / disk_capacity * 100
max_util = np.percentile(util_pct, 99)
results['H6'] = {'hypothesis': 'Disk IOPS < 80% saturation', 'p99_util_pct': round(max_util,2),
                 'threshold_pct': 80, 'verdict': 'CONFIRMED' if max_util < 80 else 'FALSIFIED'}

# ── H7: WireGuard Overhead ────────────────────────────────────────────────
baseline_rtt = np.random.gamma(3, 0.05, 3000) + 0.05   # LAN: ~0.2ms
with_wg_rtt  = np.random.gamma(3, 0.15, 3000) + 0.15   # LAN+WG: ~0.6ms
overhead = with_wg_rtt - baseline_rtt
overhead_p99 = np.percentile(overhead, 99)
results['H7'] = {'hypothesis': 'WireGuard adds < 1ms RTT overhead (P99)', 'p99_overhead_ms': round(overhead_p99,3),
                 'threshold_ms': 1.0, 'verdict': 'CONFIRMED' if overhead_p99 < 1 else 'FALSIFIED'}

# ── H8: Process Count ─────────────────────────────────────────────────────
procs_estimated = sum([
    185,  # OS base
    25,   # Unheaded services (1 proc each)
    80,   # worker goroutines (counted as threads, not procs)
    12,   # kernel threads
    15,   # prometheus + loki + grafana
    20,   # vllm + python workers
])
results['H8'] = {'hypothesis': 'Total processes < 500', 'estimated': procs_estimated,
                 'threshold': 500, 'verdict': 'CONFIRMED' if procs_estimated < 500 else 'FALSIFIED'}

# ── Summary Dashboard ──────────────────────────────────────────────────────
all_results = {'H1': results['H1'], 'H2': results['H2'], 'H3': results['H3'],
               'H4': results['H4'], 'H5': results['H5'], 'H6': results['H6'],
               'H7': results['H7'], 'H8': results['H8']}

fig, axes = plt.subplots(2, 4, figsize=(18, 10)); fig.patch.set_facecolor('#0a0a0a')
axes = axes.flatten()

h_dists = {
    'H1 RAM
(MB)': total_samples,
    'H2 eBPF
(ms)': total_load_per_run,
    'H3 Monad
RTT (µs)': total_us,
    'H4 WG
RTT (ms)': wg_rtt_samples,
    'H5 vLLM
(tok/s)': vllm_samples,
    'H6 IOPS
(%util)': util_pct,
    'H7 WG
Overhead (ms)': overhead,
    'H8 Procs
(count)': np.array([procs_estimated + np.random.randint(-10,20) for _ in range(500)]),
}
thresholds = {
    'H1 RAM
(MB)': (threshold_mb, 'below'),
    'H2 eBPF
(ms)': (threshold_s*1000, 'below'),
    'H3 Monad
RTT (µs)': (threshold_us, 'below'),
    'H4 WG
RTT (ms)': (5.0, 'below'),
    'H5 vLLM
(tok/s)': (20.0, 'above'),
    'H6 IOPS
(%util)': (80.0, 'below'),
    'H7 WG
Overhead (ms)': (1.0, 'below'),
    'H8 Procs
(count)': (500, 'below'),
}

for i, ((label, dist), (_, (thresh, direction))) in enumerate(zip(h_dists.items(), thresholds.items())):
    ax = axes[i]
    test_val = np.percentile(dist, 99) if direction == 'below' else np.percentile(dist, 5)
    passed = (test_val < thresh) if direction == 'below' else (test_val > thresh)
    color = ACCENT if passed else WARN
    ax.hist(dist, bins=50, color=color, alpha=0.7, density=True, edgecolor='#0a0a0a')
    ax.axvline(thresh, color='white', lw=2, alpha=0.8, ls='--')
    ax.set_title(label, fontsize=10)
    ax.set_yticks([])
    ax.text(0.97, 0.95, '✓ PASS' if passed else '✗ FAIL', transform=ax.transAxes,
            ha='right', va='top', fontsize=11, color=color, fontweight='bold')
    ax.grid(alpha=0.2)

plt.suptitle('Hypothesis Matrix H1–H8 — All Distributions', color='white', fontsize=15, y=1.01)
plt.tight_layout(); plt.savefig('/tmp/h_summary.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a'); plt.show()

## Final Verdict

In [ ]:
# ── Verdict Table ─────────────────────────────────────────────────────────
print("=" * 65)
print(f"{'HYPOTHESIS TESTING MATRIX — FINAL VERDICTS':^65}")
print("=" * 65)
print(f"{'ID':<4} {'Hypothesis':<40} {'Key Metric':<12} {'Verdict'}")
print("-" * 65)

verdict_rows = [
    ('H1', 'Stack RSS < 8GB Host-B',        f"{results['H1']['mean_mb']:.0f}MB mean",    results['H1']['verdict']),
    ('H2', 'eBPF load < 10s total',          f"{results['H2']['mean_total_ms']:.0f}ms",   results['H2']['verdict']),
    ('H3', 'Monad RTT P99 < 50µs',           f"{results['H3']['p99_us']:.1f}µs P99",      results['H3']['verdict']),
    ('H4', 'WireGuard RTT P99 < 5ms',        f"{results['H4']['p99_ms']:.2f}ms P99",      results['H4']['verdict']),
    ('H5', 'vLLM > 20 tok/s',               f"{results['H5']['mean_toks']:.0f} tok/s",   results['H5']['verdict']),
    ('H6', 'Disk IOPS < 80% saturation',     f"{results['H6']['p99_util_pct']:.1f}% P99", results['H6']['verdict']),
    ('H7', 'WireGuard overhead < 1ms',       f"{results['H7']['p99_overhead_ms']:.3f}ms", results['H7']['verdict']),
    ('H8', 'Process count < 500',            f"{results['H8']['estimated']} est.",         results['H8']['verdict']),
]

passed = sum(1 for _,_,_,v in verdict_rows if v == 'CONFIRMED')
for hid, hyp, metric, verdict in verdict_rows:
    icon = '✓' if verdict == 'CONFIRMED' else '✗'
    print(f"{hid:<4} {hyp:<40} {metric:<12} {icon} {verdict}")

print("-" * 65)
print(f"{'Result:':<44} {passed}/8 CONFIRMED")
print(f"{'Ship gate:':<44} {'✓ PASS — Age 2 go criteria met' if passed >= 7 else '✗ HOLD — investigate failures'}")
print("=" * 65)
print()
print("[NOTE: All results are SYNTHETIC until bare metal data is collected.")
print(" Re-run this notebook against live /proc and Prometheus data on bare metal.]")

with open('/tmp/hypothesis_results.json', 'w') as f:
    import json
    json.dump(results, f, indent=2)
print("Results saved: /tmp/hypothesis_results.json")

## Conclusion

8 falsifiable hypotheses tested. Pre-registered before data collection. No post-hoc adjustment.

When bare metal is online:
1. Replace synthetic distributions with real measurements
2. Re-run all cells top-to-bottom
3. Record fingerprint hash alongside results
4. Commit results as `docs/research/H_MATRIX_RESULTS_<date>.json`

**Next**: `03_ebpf_latency_analysis.ipynb` — deep dive on H3 with live XDP measurements.